In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import col, to_date, date_format, round as _round, row_number, desc
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_df = spark.table(bronze_table)

dedup_window = Window.partitionBy("transaction_id").orderBy(desc("ingest_time"))
 
silver_clean_df = (
    bronze_df
    .filter("user_id is not null and id is not null and amount > 0")
    .dropDuplicates(["transaction_id"])
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_time_only", date_format(col("event_time"), "HH:mm:ss"))
    .drop("event_time")
    .withColumn("_rn", row_number().over(dedup_window))
    .filter("_rn = 1")
    .drop("_rn")
    .withColumn("amount",_round(col("amount").cast("decimal(18,2)"),2))
)
 
if not spark.catalog.tableExists(silver_table):
    (silver_clean_df.write.format("delta")
        .partitionBy("event_date")
        .saveAsTable(silver_table))
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    (silver_delta.alias("target")
        .merge(
            source=silver_clean_df.alias("s"),
            condition="target.transaction_id = s.transaction_id",
        )
        .whenMatchedUpdate(set={
            "id": "s.id",
            "user_id": "s.user_id",
            "amount": "round(s.amount, 2)",
            "event_date": "s.event_date",
            "event_time_only": "s.event_time_only",
            "source_file": "s.source_file",
        })
        .whenNotMatchedInsert(values={
            "transaction_id": "s.transaction_id",
            "id": "s.id",
            "user_id": "s.user_id",
            "amount": "round(s.amount, 2)",
            "event_date": "s.event_date",
            "event_time_only": "s.event_time_only",
            "source_file": "s.source_file",
            "ingest_time": "current_timestamp()",
        })
        .execute())
 
display(spark.table(silver_table).orderBy("user_id"))